Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [1]:
NO_RECORDS = 150

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire, merge_subjects_from_new_method

d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [6]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PDNHJB', 'title': 'USA New Orleans 2005', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"><header status=""><identifier>doi:10.34894/PDNHJB</identifier><datestamp>2025-12-01T01:00:23Z</datestamp><setSpec>TUDelft</setSpec><setSpec>dataversenl</setSpec></header><metadata><resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd"><identifier identifierType="DOI">10.34894/PDNHJB</identifier><creators><creator><creatorName nameType="Personal">N/A</creatorName></creator></creators><titles><title>USA New Orleans 2005</title></titles><publisher>DataverseNL</publisher><publicationYear>2015</publicationYear><dates><date dateType="Submitted">2015-11-03</date><date dateType="Updated">2015-12-01</date></dates>

In [7]:
record_titles

['USA New Orleans 2005',
 'Beguinages',
 'Grevelingen data',
 'Metatranscriptomic census of active protists in soils',
 'Japan (City of Joso Flood)',
 'Japan',
 'Japan Practice on Levee Safety (Dr. Honjo)',
 'Xpress Access on Polaris Cycle 2018/4: XB 1890374, 1890377-8, 1890387, 1890390-2, 1990021, 1990030-40, 1990044-5, 1990062-4, 1990081',
 'Survey among Swiss Abroad',
 'Data sharing and re-use: Researcher practices, attitudes and needs - 2016',
 'Befragung junger deutschschweizer Familien zur Bedeutung von religiösen Ritualen - 2005',
 'Interest Group Preferences in Deficit Countries – Ireland, Spain and Greece',
 'ÜGK / COFO / VECOF 2023, Languages Grade 11: Keys for Data Linkage',
 'Umfrage: Immigrantinnen und Immigranten in der Schweiz-2013',
 'Sozialisationsprozesse beim Übergang in den Lehrbetrieb (SoLe) - 2011-2017',
 'Migration-Mobility Survey. Survey on living as a migrant in Switzerland',
 'Interest Group Preferences in Surplus Countries – Germany, Austria and the Netherlan

In [8]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"><header status=""><identifier>doi:10.34894/TQVCH8</identifier><datestamp>2025-12-01T01:00:24Z</datestamp><setSpec>UtrechtUniversity</setSpec><setSpec>dataversenl</setSpec></header><metadata><resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd"><identifier identifierType="DOI">10.34894/TQVCH8</identifier><creators><creator><creatorName nameType="Personal">N/A</creatorName></creator></creators><titles><title>Grevelingen data</title></titles><publisher>DataverseNL</publisher><publicationYear>2011</publicationYear><dates><date dateType="Submitted">2011-12-12</date><date dateType="Updated">2011-12-12</date></dates><resourceType resourceTypeGeneral="Dataset"/><sizes><size>367137</size><size>14848</size></sizes><formats><f

In [9]:
get_doi(record_raw_metadata[2])

'10.34894/TQVCH8'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [10]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [11]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

[]


In [12]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 39, 40, 41, 42, 43, 45, 46, 47, 49, 50, 51, 52, 53, 55, 56, 57, 59, 60, 61, 62, 65, 66, 67, 68, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 98, 99, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [13]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PDNHJB', '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PQU1NA', '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/TQVCH8', '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/XESS9X', '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YQ0L4Y', '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YURIF5', '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/T4MXVA', '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d', '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:b70e44ac-73e0-454d-a056-7a619b71b811', '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:837bfeb4-955f-4ed1-94f1-edce411ef079', '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:fa406548-fb6c-4ac7-9e64-766a78199d4c', '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:59a11e8f-5871-4ac1-a26f-29e3a02835ec', '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:3557f168-dfca-4bc3-91b4-8d5ea39f902b', '85efdee4-691d-4db2-8dd6

In [14]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [15]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, connect_ids_with_subjects_by_doi, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'N/A', 'name': None, 'surname': None, 'rank': 1, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'und', 'label': 'Undetermined'}, 'countries': [{'code': 'NL', 'label': 'Netherlands', 'provenance': None}, {'code': 'US', 'label': 'United States', 'provenance': None}], 'subjects': None, 'mainTitle': 'Beguinages', 'subTitle': None, 'descriptions': ['N/A'], 'publicationDate': '2011-01-01', 'publisher': 'DataverseNL', 'embargoEndDate': '2011-03-07', 'sources': None, 'formats': ['application/octet-stream'], 'contributors': ['website www.collective-action.info'], 'coverages': None, 'bestAccessRight': {'code': 'c_abf2', 'label': 'OPEN', 'scheme': 'http://vocabularies.coar-repositories.org/documentation/access_rights/'}, 'container': None, 'documentationUrls': None, 'codeRepositoryUrl': None, 'programmingLanguage': None, 'contactPeople': None, 'contactGroups': None, 'tools': None, 'size': '1540096', 'version': '1.

In [16]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.23662/fors-ds-1223-1', 'subjects': ['1. No poverty']},
 {'doi': '10.23662/fors-ds-1217-1',
  'subjects': ['1. No poverty', '10. No inequality']},
 {'doi': '10.23662/fors-ds-686-1',
  'subjects': ['4. Education',
   '8. Economic growth',
   '10. No inequality',
   '3. Good health']},
 {'doi': '10.48573/00a2-vw63',
  'subjects': ['4. Education',
   '8. Economic growth',
   '10. No inequality',
   '3. Good health']},
 {'doi': '10.23662/fors-ds-854-1', 'subjects': ['8. Economic growth']},
 {'doi': '10.48573/zcjj-5x79', 'subjects': ['8. Economic growth']},
 {'doi': '10.48573/c09h-h441', 'subjects': ['8. Economic growth']},
 {'doi': '10.48573/zset-sh38', 'subjects': ['8. Economic growth']},
 {'doi': '10.23662/fors-ds-675-3', 'subjects': ['8. Economic growth']},
 {'doi': '10.48573/2vz5-bh60', 'subjects': ['8. Economic growth']},
 {'doi': '10.23662/fors-ds-470-1', 'subjects': ['16. Peace & justice']},
 {'doi': '10.48573/hdc5-e885', 'subjects': ['16. Peace & justice']},
 {'doi': '1

In [17]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PDNHJB',
  'doi': '10.34894/pdnhjb'},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PQU1NA',
  'doi': '10.34894/pqu1na'},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/TQVCH8',
  'doi': '10.34894/tqvch8'},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/XESS9X',
  'doi': '10.34894/xess9x'},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YQ0L4Y',
  'doi': '10.34894/yq0l4y'},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YURIF5',
  'doi': '10.34894/yurif5'},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/T4MXVA',
  'doi': '10.34894/t4mxva'},
 {'id': '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-690-1'},
 {'id': '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:b70e44ac-73e0-454d-a056-7a619b71b811',
  'doi': '10.23662/fors-ds-849-1'},
 {'id': '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.

In [18]:
enriched_list = connect_ids_with_subjects_by_doi(id_dois, doi_subjects)
enriched_list

[{'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PDNHJB',
  'doi': '10.34894/pdnhjb',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PQU1NA',
  'doi': '10.34894/pqu1na',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/TQVCH8',
  'doi': '10.34894/tqvch8',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/XESS9X',
  'doi': '10.34894/xess9x',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YQ0L4Y',
  'doi': '10.34894/yq0l4y',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YURIF5',
  'doi': '10.34894/yurif5',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/T4MXVA',
  'doi': '10.34894/t4mxva',
  'enriched_subjects': set()},
 {'id': '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-6

In [19]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PDNHJB',
  'doi': '10.34894/pdnhjb',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PQU1NA',
  'doi': '10.34894/pqu1na',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/TQVCH8',
  'doi': '10.34894/tqvch8',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/XESS9X',
  'doi': '10.34894/xess9x',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YQ0L4Y',
  'doi': '10.34894/yq0l4y',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YURIF5',
  'doi': '10.34894/yurif5',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/T4MXVA',
  'doi': '10.34894/t4mxva',
  'enriched_subjects': set()},
 {'id': '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-6

In [20]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PDNHJB',
  'doi': '10.34894/pdnhjb',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PQU1NA',
  'doi': '10.34894/pqu1na',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/TQVCH8',
  'doi': '10.34894/tqvch8',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/XESS9X',
  'doi': '10.34894/xess9x',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YQ0L4Y',
  'doi': '10.34894/yq0l4y',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YURIF5',
  'doi': '10.34894/yurif5',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/T4MXVA',
  'doi': '10.34894/t4mxva',
  'enriched_subjects': set()},
 {'id': '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-6

In [21]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

135
135


In [22]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [23]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 8, 9, 10]

In [24]:
first_enriched

[{'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PDNHJB',
  'doi': '10.34894/pdnhjb',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/PQU1NA',
  'doi': '10.34894/pqu1na',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/TQVCH8',
  'doi': '10.34894/tqvch8',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/XESS9X',
  'doi': '10.34894/xess9x',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YQ0L4Y',
  'doi': '10.34894/yq0l4y',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/YURIF5',
  'doi': '10.34894/yurif5',
  'enriched_subjects': set()},
 {'id': '4f713a7c-aef1-4bd6-8e44-0eca1477a899::10.34894/T4MXVA',
  'doi': '10.34894/t4mxva',
  'enriched_subjects': set()},
 {'id': '85efdee4-691d-4db2-8dd6-508b0adbfcb7::oai:swissubase.ch:6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-6

In [25]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"><header status=""><identifier>doi:10.34894/PDNHJB</identifier><datestamp>2025-12-01T01:00:23Z</datestamp><setSpec>TUDelft</setSpec><setSpec>dataversenl</setSpec></header><metadata><resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd"><identifier identifierType="DOI">10.34894/PDNHJB</identifier><creators><creator><creatorName nameType="Personal">N/A</creatorName></creator></creators><titles><title>USA New Orleans 2005</title></titles><publisher>DataverseNL</publisher><publicationYear>2015</publicationYear><dates><date dateType="Submitted">2015-11-03</date><date dateType="Updated">2015-12-01</date></dates><resourceType resourceTypeGeneral="Dataset"/><sizes><size>51094</size><size>2206193</size><size>12787</size><size

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [26]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [27]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [28]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [29]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

16


[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2368238751',
  'doi': '10.15151/esrf-es-2368238751',
  'enriched_subjects': {'Raman scattering'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2306254336',
  'doi': '10.15151/esrf-es-2306254336',
  'enriched_subjects': {'Overhead (engineering)', 'Shock (circulatory)'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2307850665',
  'doi': '10.15151/esrf-es-2307850665',
  'enriched_subjects': {'Ferroelectricity'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2241571158',
  'doi': '10.15151/esrf-es-2241571158',
  'enriched_subjects': {'Metal'}},
 {'id': 'a55ca18d-8e4e-430f-8e89-04f08f2fa4c9::datapub/17',
  'doi': '10.5442/nd000017',
  'enriched_subjects': {'Hydrogen'}},
 {'id': 'a64cfb20-e461-4ec0-a069-a00c1fde6c64::704',
  'doi': '10.14278/rodare.704',
  'enriched_subjects': {'Graphene',
   'Magneto',
   'Nonlinear Optical Materials Studies',
   'Plasmon',
   'Plasmonic and Surface Plasmon Research',
   'Thermal Radiati

In [30]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.7
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()}]

In [31]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

In [32]:
merge_subjects_from_new_method(enriched_results, oaire_results)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

In [33]:
## DATACITE REST API

DATACITE_API_URL = "https://api.datacite.org/dois"

DataCite provides an additional "enriched" option for fetching doi-by-doi, could be interesting.

In [34]:
def _datacite_batch_doi_resolve(dois: list[str]) -> list[dict]:
    all_results = []

    clean_dois = [
        doi.replace("https://doi.org/", "")
           .replace("http://doi.org/", "")
        for doi in dois
    ]

    for doi in clean_dois:
        response = requests.get(
            f"{DATACITE_API_URL}/{doi}",
            params={"disable-facets": "true"},
            timeout=30,
        )
        if response.status_code == 404:
            continue
        response.raise_for_status()
        all_results.append(response.json().get("data", {}))

    return all_results

In [35]:
DC_res = _datacite_batch_doi_resolve(doi_to_enrich)

In [36]:
DC_res

[{'id': '10.34894/pdnhjb',
  'type': 'dois',
  'attributes': {'doi': '10.34894/pdnhjb',
   'prefix': '10.34894',
   'suffix': 'pdnhjb',
   'identifiers': [{'identifier': '10411/20604', 'identifierType': 'Handle'}],
   'alternateIdentifiers': [{'alternateIdentifierType': 'Handle',
     'alternateIdentifier': '10411/20604'}],
   'creators': [{'name': ':unav', 'affiliation': [], 'nameIdentifiers': []}],
   'titles': [{'title': 'USA New Orleans 2005'}],
   'publisher': 'DataverseNL',
   'container': {},
   'publicationYear': 2015,
   'subjects': [],
   'contributors': [],
   'dates': [{'date': '2015-11-03', 'dateType': 'Submitted'},
    {'date': '2015-11-03', 'dateType': 'Available'},
    {'date': '2015-12-01', 'dateType': 'Updated'},
    {'date': '2015', 'dateType': 'Issued'}],
   'language': None,
   'types': {'ris': 'DATA',
    'bibtex': 'misc',
    'citeproc': 'dataset',
    'schemaOrg': 'Dataset',
    'resourceTypeGeneral': 'Dataset'},
   'relatedIdentifiers': [],
   'relatedItems': [

In [37]:
[el['attributes']['subjects'] for el in DC_res]

[[],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [{'subject': 'Life Sciences', 'subjectScheme': 'Proposal Type Description'},
  {'subject': 'LS-3137', 'subjectScheme': 'Proposal'},
  {'subject': 'ID16A', 'subjectScheme': 'Instrument'}],
 [{'subject': 'Macromolecular Crystallography',
   'subjectScheme': 'Proposal Type Description'},
  {'subject': 'MX-2655', 'subjectScheme': 'Proposal'},
  {'subject': 'ID30A-3', 'subjectScheme': 'Instrument'}],
 [{'subject': 'Chemistry', 'subjectScheme': 'Proposal Type Description'},
  {'subject': 'CH-8121', 'subjectScheme': 'Proposal'},
  {'subject': 'ID20', 'subjectScheme': 'Instrument'},
  {'subject': 'XRAYS, X-ray Radiation Technique',
   'valueUri': 'https://w3id.org/PaN/ESRFET#XRAYS',
   'schemeUri': 'https://w3id.org/PaN/ESRFET/',
   'subjectScheme': 'ESRFET',
   'classificationCode': 'ESRFET#XRAYS'}],
 [{'subject': 'Medicine', 'subjectScheme': 'Proposal Type Description'},
  {'subject

In [51]:
slist = [el['attributes']['subjects'] for el in DC_res]

[[el.get('subject') for el in sublist] for sublist in slist]

[[],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 ['Life Sciences', 'LS-3137', 'ID16A'],
 ['Macromolecular Crystallography', 'MX-2655', 'ID30A-3'],
 ['Chemistry', 'CH-8121', 'ID20', 'XRAYS, X-ray Radiation Technique'],
 ['Medicine', 'MD-1364', 'ID21'],
 ['Earth Sciences', 'ES-1529', 'ID14', 'x-ray probe'],
 ['Applied Material Science', 'MA-6246', 'ID01', 'x-ray probe'],
 ['Chemistry', 'CH-6071', 'ID15A'],
 ['Earth Sciences', 'ES-1517', 'BM23', 'x-ray probe'],
 ['Cultural Heritage', 'HG-203', 'ID15A'],
 ['Macromolecular Crystallography', 'MX-2746', 'ID23-2'],
 ['Soft Condensed Matter Science', 'SC-5508', 'BM28', 'x-ray probe'],
 ['Macromolecular Crystallography', 'MX-2653', 'ID23-2', 'x-ray probe'],
 ['Macromolecular Crystallography', 'MX-2584', 'ID30A-3', 'x-ray probe'],
 ['Soft Condensed Matter Science', 'SC-5581', 'ID15A'],
 ['Method and Instrumentation', 'MI-1523', 'ID19'],
 ['Chemistry', 'CH-6642', 'ID16A'],
 [None, 'A01-

In [54]:
def _datacite_get_doi_subjects(response: list[dict]):
    """
    Transform API response into a list of DOIs and their subject-sets.

    OpenAlex responses have 3 machine-derived fields of interest:
    topics, keywords, concepts

    Concepts are a legacy version of keywords so we'll skip those.
    We'll consider keywords and concepts as subjects of interest. We'll further filter them out by similarity score.
    
    :param response: List[dict], the API response
    :param score_treshold: float, minimum similarity score a topic/keyword must have to be included as a subject
    :return: List[dict]
        dict:
            DOI - DOI that is linked to at least one subject
            [Subjects] - all subjects for the given DOI
    """
    if response == []:
        # no DOIs with any results
        return []
    
    # sets of DOIs with relevant sets of subjects. Later, spread to individual DOIs
    enriched_dois = []
    enriched_subjects = []

    for el in response:
        sub_list = el['attributes']['subjects']

        if (sub_list == []):
            # this element doesn't contribute to enriching subjects
            continue

        # It's possible that some dicts do not have the subject key
        subjects = [item.get('subject') for item in sub_list]

        if not subjects:
            continue
        
        current_doi = el['id']

        # Strip DOI from prefixes and make it lowercase
        current_doi = current_doi.replace("https://doi.org/", "").replace("http://doi.org/", "").lower()

        enriched_dois.append(current_doi)

        enriched_subjects.append(subjects)

    # Create a list of dicts as output

    doi_subjects = []

    for ix in range(len(enriched_dois)):
        doi_sub_pair = {'doi': enriched_dois[ix],
                        'subjects': enriched_subjects[ix]}
        doi_subjects.append(doi_sub_pair)

    return doi_subjects

In [55]:
_datacite_get_doi_subjects(DC_res)

[{'doi': '10.15151/esrf-es-1038848186',
  'subjects': ['Life Sciences', 'LS-3137', 'ID16A']},
 {'doi': '10.15151/esrf-es-2114669173',
  'subjects': ['Macromolecular Crystallography', 'MX-2655', 'ID30A-3']},
 {'doi': '10.15151/esrf-es-2368238751',
  'subjects': ['Chemistry',
   'CH-8121',
   'ID20',
   'XRAYS, X-ray Radiation Technique']},
 {'doi': '10.15151/esrf-es-1163203414',
  'subjects': ['Medicine', 'MD-1364', 'ID21']},
 {'doi': '10.15151/esrf-es-2008566834',
  'subjects': ['Earth Sciences', 'ES-1529', 'ID14', 'x-ray probe']},
 {'doi': '10.15151/esrf-es-2000217707',
  'subjects': ['Applied Material Science', 'MA-6246', 'ID01', 'x-ray probe']},
 {'doi': '10.15151/esrf-es-581537692',
  'subjects': ['Chemistry', 'CH-6071', 'ID15A']},
 {'doi': '10.15151/esrf-es-1830135906',
  'subjects': ['Earth Sciences', 'ES-1517', 'BM23', 'x-ray probe']},
 {'doi': '10.15151/esrf-es-1220324147',
  'subjects': ['Cultural Heritage', 'HG-203', 'ID15A']},
 {'doi': '10.15151/esrf-es-2359223465',
  'subje

# Related DOIs  
Extract related identifiers that are also DOIs. If they are in the database, use their subjects. If not, use DOI-resolving methods on them.

Note:
a DOI must start with 10. prefix. I've seen some related DOIs that start with 0.  , if I add a 1 to the start it resolves properly.

## SQL query to see records with related DOIs   
```
--- get records with related identifiers of any tipe
WITH query AS (
    SELECT
        raw_metadata,
		doi as record_doi,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 1000;
```

47k records have SOME relatedIdentifiers - lots of them ISSN or proprietary ids

In [38]:
# Let's try out some records with related DOIs, for testing we won't limit ourselves to records with no raw_subjects.

query_related_dois = """
WITH query AS (
    SELECT
        id,
        raw_metadata,
		doi as record_doi,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 100;
"""

In [39]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_related_dois)
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_subjects = [row["subjects"] for row in results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]
        record_related_dois = [row["related"] for row in results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '557d465d-8e79-45ed-9e44-60e105fb0ecb::irb_829', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:dabar.srce.hr:irb_829</identifier>\n    <datestamp>2026-01-14</datestamp>\n  </header>\n  <metadata>\n    <datacite:resource xmlns:datacite="http://datacite.org/schema/kernel-4" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dabar="http://dabar.srce.hr/standards/schema/1.0" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.4/metadata.xsd">\n      <datacite:identifier identifierType="URL">https://urn.nsk.hr/urn:nbn:hr:241:759840</datacite:identifier>\n      <datacite:alternateIdentifiers>\n        <datacite:alternateIdentifier alternateIdentifierType="LOCAL">irb:829</datacite:alternateIdentifier>\n        <datacite:alternateIdentifier alternateIdentifierType="URN">urn:nbn:hr:241:759840</datacite:alternateIdentifier>\

In [40]:
results

[{'id': '557d465d-8e79-45ed-9e44-60e105fb0ecb::irb_829',
  'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:dabar.srce.hr:irb_829</identifier>\n    <datestamp>2026-01-14</datestamp>\n  </header>\n  <metadata>\n    <datacite:resource xmlns:datacite="http://datacite.org/schema/kernel-4" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dabar="http://dabar.srce.hr/standards/schema/1.0" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.4/metadata.xsd">\n      <datacite:identifier identifierType="URL">https://urn.nsk.hr/urn:nbn:hr:241:759840</datacite:identifier>\n      <datacite:alternateIdentifiers>\n        <datacite:alternateIdentifier alternateIdentifierType="LOCAL">irb:829</datacite:alternateIdentifier>\n        <datacite:alternateIdentifier alternateIdentifierType="URN">urn:nbn:hr:241:759840</datacite:alternateIdentifier

In [41]:
# For a specific DOI set, search our datasetdb and retrieve subjects if any.
# Maybe a record references/is part of another dataset so the subjects are relevant too.

# This query works for a list of DOIs -  if a record has multiples

query_subjects_by_doi = """
WITH query AS (
    SELECT
    id,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects
    FROM public.records
)
SELECT subjects
FROM query
WHERE query.id = ANY(%s)
LIMIT 100;
"""

In [42]:
first_related_doi = record_related_dois[0]
first_related_doi

['10.48550/arXiv.2601.08451']

In [43]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_subjects_by_doi, (first_related_doi,))
        
        record_res = cur.fetchall()
except:
    print(f"Query error: {e}")
        

record_res

[]

In [44]:
# Go through all test cases

related_subjects = []

for i in range(len(record_ids)):
    try:
        with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
            cur = conn.cursor()
            related_dois = record_related_dois[i]
            cur.execute(query_subjects_by_doi, (related_dois,))
            
            record_res = cur.fetchall()
            related_subjects.append(record_res)
    except:
        print(f"Query error: {e}")
        

In [45]:
# No hits
[res for res in related_subjects if res]

[]

In [46]:
# Extra: go through our DB
# and ALSO retrieve all subjects from openaire and openalex for those individual DOIs

related_subjects = []

for i in range(len(record_ids)):
    try:
        with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
            cur = conn.cursor()
            related_dois = record_related_dois[i]
            cur.execute(query_subjects_by_doi, (related_dois,))
            
            record_res = cur.fetchall()
            record_res = set(record_res)
            # We need an id-doi pairing for each doi, we'll make fake ids since we won't use them for joining later
            id_dois = [{'id': str(i), 'doi': doi,} for i, doi in enumerate(related_dois)]
            print(id_dois)

            openaire = batch_enrich_subjects_from_openaire(id_dois)
            openaire_subjects = set().union(*(el["enriched_subjects"] for el in openaire))
            
            openalex = batch_enrich_subjects_from_openalex(id_dois, OpenAlex_API_key)
            openalex_subjects = set().union(*(el["enriched_subjects"] for el in openalex))

            # Union of all three
            record_res = record_res | openaire_subjects | openalex_subjects

            related_subjects.append(record_res)
    except:
        print(f"Query error: {e}")
        

[{'id': '0', 'doi': '10.48550/arXiv.2601.08451'}]
[{'id': '0', 'doi': '10.1108/JPBM-12-2022-4263'}]
[{'id': '0', 'doi': '10.26481/dis.20260330ri'}]
[{'id': '0', 'doi': '10.1007/s00723-024-01693-5'}]
[{'id': '0', 'doi': 'doi.org/10.3998/jep.8085'}]
[{'id': '0', 'doi': '10.21827/whv8m045'}]
[{'id': '0', 'doi': '10.1103/bvgw-9q78'}]
[{'id': '0', 'doi': '10.1007/s10519-026-10261-5'}, {'id': '1', 'doi': '10.64898/2025.12.18.695080'}]
[{'id': '0', 'doi': '10.7554/eLife.98340.2'}]
[{'id': '0', 'doi': '10.1017/ssh.2020.13'}]
[{'id': '0', 'doi': '10.1111/bjep.70088'}]
[{'id': '0', 'doi': '10.1039/D4DT02519A'}]
[{'id': '0', 'doi': '10.3390/medicina61071154'}]
[{'id': '0', 'doi': '10.1111/jace.19911'}]
[{'id': '0', 'doi': 'https://doi.org/10.34965/i10518'}]
[{'id': '0', 'doi': 'https://doi.org/10.34965/enodo.i60393'}]
[{'id': '0', 'doi': 'https://doi.org/10.34965/i10122'}]
[{'id': '0', 'doi': 'https://doi.org/10.34965/i8865'}]
[{'id': '0', 'doi': 'https://doi.org/10.34965/i10638'}]
[{'id': '0', '

In [47]:
related_subjects

[{'Ambiguity',
  'Constraint (computer-aided design)',
  'FOS: Physical sciences',
  'High Energy Physics - Phenomenology',
  'High Energy Physics - Phenomenology (hep-ph)'},
 {'Advertising',
  'Brand Relationships',
  'Brand awareness',
  'Brand equity',
  'Brand management',
  'Brand relationships',
  'Brands',
  'Conceptualization',
  'Construct (python library)',
  'Consumer Behavior in Brand Consumption and Identification',
  'Consumer Brand relationships',
  'Consumer brand relationships',
  'Consumer psychology',
  'Customer Service Quality and Loyalty',
  'Digital Marketing and Social Media',
  'Marketing',
  'Originality',
  'Perspective (graphical)',
  'Prototype analysis',
  'Value (mathematics)',
  'brands',
  'prototype analysis'},
 set(),
 {'B1 estimation',
  'CW EPR',
  'Electron Spin Resonance Studies',
  'Electron paramagnetic resonance',
  'Lanthanide and Transition Metal Complexes',
  'Magnetism in coordination complexes',
  'Nuclear magnetic resonance',
  'Omega',
 